# Python + SQL Workbook
**CUN · Área de Datos & Business Intelligence**

Plantilla para exploración, consultas y transformaciones sobre SQL Server.

## 1. Dependencias

In [ ]:
# Instalar si es necesario
# %pip install pandas sqlalchemy pyodbc python-dotenv

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(override=True)
print("Librerías cargadas correctamente.")

## 2. Conexión a SQL Server

In [ ]:
# ── Parámetros de conexión ────────────────────────────────────────────
DB_SERVER   = os.getenv("ETL_DB_SERVER",   "172.16.1.33")
DB_DATABASE = os.getenv("ETL_DB_DATABASE", "CUN_REPOSITORIO")
DB_USER     = os.getenv("ETL_DB_USER",     "")
DB_PASSWORD = os.getenv("ETL_DB_PASSWORD", "")

# Si no hay .env, puedes escribir las credenciales aquí temporalmente:
# DB_USER     = "tu_usuario"
# DB_PASSWORD = "tu_contraseña"

def get_engine(driver="ODBC Driver 18 for SQL Server"):
    conn_str = (
        f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}@{DB_SERVER}/{DB_DATABASE}"
        f"?driver={driver.replace(' ', '+')}&TrustServerCertificate=yes"
    )
    return create_engine(conn_str, fast_executemany=True)

engine = get_engine()

# Test de conexión
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION")).scalar()
    print(result[:80])

## 3. Consultas SQL → DataFrame

In [ ]:
# Función utilitaria
def sql(query: str, params: dict = None) -> pd.DataFrame:
    """Ejecuta una consulta y retorna un DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

print("Función sql() lista.")

In [ ]:
# ── crm.Registros_CRM · 2026 ──────────────────────────────────────────
df_leads = sql("""
    SELECT *
      FROM crm.Registros_CRM
     WHERE FechaCreación >= '2026-01-01'
       AND FechaCreación <  '2027-01-01'
""")

print(f"Filas: {len(df_leads):,}  |  Columnas: {df_leads.shape[1]}")
df_leads.head()

In [ ]:
# ── zoho.Base_Personas · 2026 · Nuevo · Pregrado/Posgrado ─────────────
df_zoho = sql("""
    SELECT *
      FROM zoho.Base_Personas
     WHERE ANIO = '2026'
       AND CLASE_ACTUAL = 'Nuevo'
       AND TIPO IN ('Pregrado', 'Posgrado')
""")

print(f"Filas: {len(df_zoho):,}  |  Columnas: {df_zoho.shape[1]}")
df_zoho.head()

## 4. Full Outer Join · CRM ↔ Zoho

**Estrategia de llaves:**
| Prioridad | Llave CRM | Llave Zoho | Confiabilidad |
|-----------|-----------|------------|---------------|
| 1° (principal) | `Número_de_Documento` + `Periodo` | `DOC_ALUM` + `PERIODO` | Alta — ID único y estable |
| 2° (fallback) | `Teléfono` + `Periodo` | `TEL_CASA` + `PERIODO` | Media — requiere normalización |

El join aplica primero la llave por documento; los registros no emparejados pasan a una segunda ronda por teléfono.

In [ ]:
# ── 4.1 Diagnóstico de llaves ─────────────────────────────────────────
print("=== CRM · df_leads ===")
for col in ["Teléfono", "Número_de_Documento", "Periodo"]:
    if col in df_leads.columns:
        nulos = df_leads[col].isna().sum()
        uniq  = df_leads[col].nunique()
        print(f"  {col:30s} | nulos={nulos:>6,} ({nulos/len(df_leads)*100:.1f}%)  | únicos={uniq:>6,}")
    else:
        print(f"  {col:30s} | *** COLUMNA NO ENCONTRADA ***")

print("\n=== Zoho · df_zoho ===")
for col in ["TEL_CASA", "DOC_ALUM", "PERIODO"]:
    if col in df_zoho.columns:
        nulos = df_zoho[col].isna().sum()
        uniq  = df_zoho[col].nunique()
        print(f"  {col:30s} | nulos={nulos:>6,} ({nulos/len(df_zoho)*100:.1f}%)  | únicos={uniq:>6,}")
    else:
        print(f"  {col:30s} | *** COLUMNA NO ENCONTRADA ***")

In [ ]:
# ── 4.2 Funciones de limpieza y normalización ─────────────────────────
import re

def clean_phone(val) -> str | None:
    """
    Limpieza de teléfono colombiano:
    1. Elimina espacios y caracteres especiales (conserva solo dígitos)
    2. Busca el primer '3' de izquierda a derecha
    3. Extrae los 10 caracteres siguientes (incluido el '3')
    Retorna None si no cumple el formato esperado.
    """
    if pd.isna(val) or str(val).strip() == "":
        return None
    digits = re.sub(r"[^\d]", "", str(val))   # solo dígitos
    idx = digits.find("3")                     # primer '3'
    if idx == -1:
        return None
    extracted = digits[idx : idx + 10]
    return extracted if len(extracted) == 10 else None


def clean_doc(val) -> str | None:
    """
    Normalización de número de documento:
    Elimina espacios, guiones y caracteres especiales; convierte a mayúsculas.
    """
    if pd.isna(val) or str(val).strip() == "":
        return None
    return re.sub(r"[^\w]", "", str(val)).strip().upper()


def clean_periodo(val) -> str | None:
    """Normaliza periodo a string limpio (ej: '20251', '2025-1' → '20251')."""
    if pd.isna(val) or str(val).strip() == "":
        return None
    return re.sub(r"[^0-9]", "", str(val)).strip()


# ── Aplicar limpieza a CRM ────────────────────────────────────────────
crm = df_leads.copy()
crm["_tel_key"]  = crm["Teléfono"].apply(clean_phone)
crm["_doc_key"]  = crm["Número_de_Documento"].apply(clean_doc)
crm["_per_key"]  = crm["Periodo"].apply(clean_periodo)

# ── Aplicar limpieza a Zoho ───────────────────────────────────────────
zoho = df_zoho.copy()
zoho["_tel_key"] = zoho["TEL_CASA"].apply(clean_phone)
zoho["_doc_key"] = zoho["DOC_ALUM"].apply(clean_doc)
zoho["_per_key"] = zoho["PERIODO"].apply(clean_periodo)

print("Limpieza aplicada.")
print(f"\nCRM  · teléfonos válidos : {crm['_tel_key'].notna().sum():,} / {len(crm):,}")
print(f"CRM  · documentos válidos: {crm['_doc_key'].notna().sum():,} / {len(crm):,}")
print(f"Zoho · teléfonos válidos : {zoho['_tel_key'].notna().sum():,} / {len(zoho):,}")
print(f"Zoho · documentos válidos: {zoho['_doc_key'].notna().sum():,} / {len(zoho):,}")

In [ ]:
# ── 4.3 Full Outer Join · Estrategia de doble llave ──────────────────
#
#  Ronda 1 — Llave principal: DOC + PERIODO (más confiable)
#  Ronda 2 — Llave fallback : TEL + PERIODO  (solo para no emparejados)
#  Resultado: unión de ambas rondas + registros que no emparejaron en ninguna
# ─────────────────────────────────────────────────────────────────────

crm_cols  = [c for c in crm.columns  if not c.startswith("_")]
zoho_cols = [c for c in zoho.columns if not c.startswith("_")]

# -- Ronda 1: join por Documento + Periodo --------------------------------
r1 = crm.merge(
    zoho,
    left_on=["_doc_key", "_per_key"],
    right_on=["_doc_key", "_per_key"],
    how="outer",
    suffixes=("_crm", "_zoho"),
    indicator=True,
).rename(columns={"_merge": "_match_type"})

r1["_match_type"] = r1["_match_type"].map({
    "both"      : "doc+periodo",
    "left_only" : "solo_crm",
    "right_only": "solo_zoho",
})

# Separar los emparejados en ronda 1 de los que quedaron sueltos
r1_matched   = r1[r1["_match_type"] == "doc+periodo"].copy()
crm_unmatched  = crm[crm.index.isin(r1[r1["_match_type"] == "solo_crm"].index)]
zoho_unmatched = zoho[zoho.index.isin(r1[r1["_match_type"] == "solo_zoho"].index)]

print(f"Ronda 1 (doc+periodo):")
print(f"  Emparejados : {len(r1_matched):>6,}")
print(f"  Solo CRM    : {len(crm_unmatched):>6,}")
print(f"  Solo Zoho   : {len(zoho_unmatched):>6,}")

# -- Ronda 2: join por Teléfono + Periodo (sobre los no emparejados) ------
r2 = crm_unmatched.merge(
    zoho_unmatched,
    left_on=["_tel_key", "_per_key"],
    right_on=["_tel_key", "_per_key"],
    how="outer",
    suffixes=("_crm", "_zoho"),
    indicator=True,
).rename(columns={"_merge": "_match_type"})

r2["_match_type"] = r2["_match_type"].map({
    "both"      : "tel+periodo",
    "left_only" : "solo_crm",
    "right_only": "solo_zoho",
})

print(f"\nRonda 2 (tel+periodo, sobre no emparejados):")
print(f"  Emparejados : {r2['_match_type'].eq('tel+periodo').sum():>6,}")
print(f"  Solo CRM    : {r2['_match_type'].eq('solo_crm').sum():>6,}")
print(f"  Solo Zoho   : {r2['_match_type'].eq('solo_zoho').sum():>6,}")

# -- Unión final ----------------------------------------------------------
df_join = pd.concat([r1_matched, r2], ignore_index=True)

# Eliminar columnas auxiliares de llaves normalizadas
df_join.drop(columns=[c for c in df_join.columns if c.startswith("_") and c != "_match_type"],
             inplace=True, errors="ignore")

print(f"\n{'─'*45}")
print(f"RESULTADO FINAL · df_join")
print(f"  Total filas     : {len(df_join):>6,}")
print(f"  doc+periodo     : {df_join['_match_type'].eq('doc+periodo').sum():>6,}")
print(f"  tel+periodo     : {df_join['_match_type'].eq('tel+periodo').sum():>6,}")
print(f"  Solo CRM        : {df_join['_match_type'].eq('solo_crm').sum():>6,}")
print(f"  Solo Zoho       : {df_join['_match_type'].eq('solo_zoho').sum():>6,}")
print(f"  Columnas totales: {df_join.shape[1]:>6,}")
df_join.head()

## 5. Consulta · coe.venta_contact_nuevo

In [ ]:
# ── coe.venta_contact_nuevo ───────────────────────────────────────────
df_llamadas = sql("""
    SELECT *
      FROM coe.venta_contact_nuevo
""")

print(f"Filas: {len(df_llamadas):,}  |  Columnas: {df_llamadas.shape[1]}")
df_llamadas.head()

## 6. Análisis de llave · df_leads ↔ df_llamadas

Se prueban todas las combinaciones `Teléfono_{leads} & {fecha_leads}` vs `Celular_Prospecto & Fecha`.  
La llave ganadora maximiza **matches** y minimiza **nulos en la llave**.

In [ ]:
# ── 6.1 Diagnóstico de llave · df_leads ↔ df_llamadas ────────────────
import re, pandas as pd

# Reutiliza clean_phone si ya fue definida arriba; si no, se redefine aquí
def _clean_phone(val):
    if pd.isna(val) or str(val).strip() == "":
        return None
    digits = re.sub(r"[^\d]", "", str(val))
    idx = digits.find("3")
    if idx == -1:
        return None
    extracted = digits[idx: idx + 10]
    return extracted if len(extracted) == 10 else None

def _to_date(series):
    """Convierte cualquier columna a date (sin hora) para comparación."""
    try:
        return pd.to_datetime(series, errors="coerce").dt.normalize()
    except Exception:
        return pd.Series([pd.NaT] * len(series), index=series.index)

# ── Columnas fecha candidatas en df_leads ─────────────────────────────
DATE_COLS_LEADS = [
    "FEC_CREA", "FEC_MOD", "FechaCreación", "FechaUltimaActividad",
    "FechaModificacion", "fecha", "FechaModificacionHis",
    "FechaModificacionHoraHis", "FEC_MODIFI", "FechaContacto",
]
DATE_COL_LLAMADAS = "Fecha"
TEL_LEADS    = "Teléfono"
TEL_LLAMADAS = "Celular_Prospecto"

# Verificar columnas disponibles
existing_dates = [c for c in DATE_COLS_LEADS if c in df_leads.columns]
missing_dates  = [c for c in DATE_COLS_LEADS if c not in df_leads.columns]

print(f"Columnas fecha encontradas en df_leads : {existing_dates}")
print(f"Columnas fecha NO encontradas           : {missing_dates}")
print(f"'{TEL_LEADS}' en df_leads    : {TEL_LEADS in df_leads.columns}")
print(f"'{TEL_LLAMADAS}' en df_llamadas: {TEL_LLAMADAS in df_llamadas.columns}")
print(f"'{DATE_COL_LLAMADAS}' en df_llamadas : {DATE_COL_LLAMADAS in df_llamadas.columns}")
print()

# ── Preparar llave de teléfono en df_llamadas ─────────────────────────
ll = df_llamadas.copy()
ll["_tel"] = ll[TEL_LLAMADAS].apply(_clean_phone)
ll["_fecha"] = _to_date(ll[DATE_COL_LLAMADAS])

ll_key_valid = ll.dropna(subset=["_tel", "_fecha"])
print(f"df_llamadas · filas con tel+fecha válidos: {len(ll_key_valid):,} / {len(ll):,}\n")

# ── Probar cada combinación de fecha en df_leads ──────────────────────
resultados = []

leads_base = df_leads.copy()
leads_base["_tel"] = leads_base[TEL_LEADS].apply(_clean_phone) if TEL_LEADS in leads_base.columns else None

for col in existing_dates:
    leads_base[f"_date_{col}"] = _to_date(leads_base[col])
    key_valid = leads_base.dropna(subset=["_tel", f"_date_{col}"])

    merged = key_valid.merge(
        ll_key_valid[["_tel", "_fecha"]].drop_duplicates(),
        left_on=["_tel", f"_date_{col}"],
        right_on=["_tel", "_fecha"],
        how="inner",
    )

    nulos_tel  = leads_base["_tel"].isna().sum()
    nulos_date = leads_base[f"_date_{col}"].isna().sum()
    matches    = len(merged)
    cobertura  = matches / len(leads_base) * 100 if len(leads_base) > 0 else 0

    resultados.append({
        "fecha_leads"  : col,
        "nulos_tel"    : nulos_tel,
        "nulos_fecha"  : nulos_date,
        "matches"      : matches,
        "cobertura_%"  : round(cobertura, 2),
    })

df_resultado = (
    pd.DataFrame(resultados)
    .sort_values("matches", ascending=False)
    .reset_index(drop=True)
)

print("=== Ranking de combinaciones Teléfono & {fecha} ===")
display(df_resultado)

mejor = df_resultado.iloc[0]
print(f"\n✔ Mejor llave: Teléfono & {mejor['fecha_leads']}")
print(f"  Matches    : {mejor['matches']:,}  ({mejor['cobertura_%']}% de df_leads)")

In [ ]:
# ── 6.2 Muestra top 100 · Full Outer Join df_leads ↔ df_llamadas ──────
# Usa automáticamente la mejor llave encontrada en 6.1

mejor_fecha = df_resultado.iloc[0]["fecha_leads"]   # columna fecha ganadora
print(f"Llave usada: Teléfono & {mejor_fecha}  ↔  Celular_Prospecto & Fecha\n")

# Preparar lados del join con claves normalizadas
_leads = df_leads.copy()
_leads["_tel"]   = _leads["Teléfono"].apply(_clean_phone)
_leads["_fecha"] = _to_date(_leads[mejor_fecha])

_ll = df_llamadas.copy()
_ll["_tel"]   = _ll["Celular_Prospecto"].apply(_clean_phone)
_ll["_fecha"] = _to_date(_ll["Fecha"])

# Full outer join
_joined = _leads.merge(
    _ll,
    on=["_tel", "_fecha"],
    how="outer",
    suffixes=("_leads", "_llamadas"),
)

# Seleccionar columnas de visualización
# Resuelve nombres con sufijo si hay colisión
def _col(df, name, sufijos=("_leads", "_llamadas")):
    if name in df.columns:
        return name
    for s in sufijos:
        if name + s in df.columns:
            return name + s
    return None

cols_wanted = {
    "Usuario_Modificacion" : _col(_joined, "Usuario_Modificacion"),
    "Teléfono"             : _col(_joined, "Teléfono"),
    f"{mejor_fecha}"       : _col(_joined, mejor_fecha),
    "Celular_Prospecto"    : _col(_joined, "Celular_Prospecto"),
    "Fecha"                : _col(_joined, "Fecha"),
    "Asesor"               : _col(_joined, "Asesor"),
    "Correo"               : _col(_joined, "Correo"),
}

# Filtrar solo las que existen
cols_display = {alias: real for alias, real in cols_wanted.items() if real}
missing_cols = [alias for alias, real in cols_wanted.items() if not real]
if missing_cols:
    print(f"⚠ Columnas no encontradas: {missing_cols}")

muestra = (
    _joined[list(cols_display.values())]
    .rename(columns={v: k for k, v in cols_display.items()})
    .head(100)
)

print(f"Total filas del join: {len(_joined):,}")
muestra

## 7. Full Outer Join · df_leads ↔ df_zoho ↔ df_llamadas (llave: Teléfono)

In [ ]:
import re

# ── Función de limpieza de teléfono (reutilizable) ────────────────────
def norm_tel(val) -> str | None:
    """Extrae 10 dígitos colombianos: elimina no-dígitos, busca primer '3', toma 10."""
    if val is None or (isinstance(val, float) and __import__('math').isnan(val)):
        return None
    digits = re.sub(r"\D", "", str(val))
    idx = digits.find("3")
    if idx == -1:
        return None
    chunk = digits[idx: idx + 10]
    return chunk if len(chunk) == 10 else None

def _to_date(series):
    """Convierte cualquier columna a date (sin hora) para comparación."""
    try:
        return pd.to_datetime(series, errors="coerce").dt.normalize()
    except Exception:
        return pd.Series([pd.NaT] * len(series), index=series.index)

# ── Llave fecha ganadora (viene de Sección 6.1) ───────────────────────
mejor_fecha_lvl3 = df_resultado.iloc[0]["fecha_leads"]
print(f"Llave fecha usada (Nivel 2→3): Teléfono & {mejor_fecha_lvl3}\n")

# ── Copias con llaves normalizadas ────────────────────────────────────
_leads    = df_leads.copy()
_zoho     = df_zoho.copy()
_llamadas = df_llamadas.copy()

_leads["_tel"]      = _leads["Teléfono"].apply(norm_tel)
_leads["_fecha"]    = _to_date(_leads[mejor_fecha_lvl3])
_zoho["_tel"]       = _zoho["TEL_CASA"].apply(norm_tel)
_llamadas["_tel"]   = _llamadas["Celular_Prospecto"].apply(norm_tel)
_llamadas["_fecha"] = _to_date(_llamadas["Fecha"])

print(f"df_leads    · tels válidos  : {_leads['_tel'].notna().sum():>6,} / {len(_leads):,}")
print(f"df_leads    · fechas válidas: {_leads['_fecha'].notna().sum():>6,} / {len(_leads):,}")
print(f"df_zoho     · tels válidos  : {_zoho['_tel'].notna().sum():>6,} / {len(_zoho):,}")
print(f"df_llamadas · tels válidos  : {_llamadas['_tel'].notna().sum():>6,} / {len(_llamadas):,}")
print(f"df_llamadas · fechas válidas: {_llamadas['_fecha'].notna().sum():>6,} / {len(_llamadas):,}")

# ── Ronda 1: df_leads FULL OUTER JOIN df_zoho · llave: _tel ──────────
r1 = _leads.merge(
    _zoho,
    on="_tel",
    how="outer",
    suffixes=("_leads", "_zoho"),
)

# ── Ronda 2: resultado FULL OUTER JOIN df_llamadas · llave: _tel + _fecha ──
df_cubos_leads_zoho_llamadas = r1.merge(
    _llamadas,
    on=["_tel", "_fecha"],
    how="outer",
    suffixes=("", "_llamadas"),
)

# Eliminar columnas auxiliares
df_cubos_leads_zoho_llamadas.drop(
    columns=[c for c in ["_tel", "_fecha"] if c in df_cubos_leads_zoho_llamadas.columns],
    inplace=True,
)

# ── Resumen ───────────────────────────────────────────────────────────
total = len(df_cubos_leads_zoho_llamadas)
print(f"\ndf_cubos_leads_zoho_llamadas")
print(f"  Filas   : {total:,}")
print(f"  Columnas: {df_cubos_leads_zoho_llamadas.shape[1]:,}")

df_cubos_leads_zoho_llamadas.head()

In [ ]:
# ── Nulos por columna ─────────────────────────────────────────────────
nulos = df_leads.isnull().sum()
nulos_pct = (nulos / len(df_leads) * 100).round(2)

resumen_nulos = (
    pd.DataFrame({"nulos": nulos, "pct_%": nulos_pct})
    .query("nulos > 0")
    .sort_values("pct_%", ascending=False)
)

if resumen_nulos.empty:
    print("Sin valores nulos.")
else:
    display(resumen_nulos)

In [ ]:
# ── Estadísticas descriptivas ─────────────────────────────────────────
df_leads.describe(include="all").T

In [ ]:
# ── Duplicados ────────────────────────────────────────────────────────
dups = df_leads.duplicated().sum()
print(f"Filas duplicadas: {dups:,} ({dups/len(df_leads)*100:.2f}%)")